# WebLog Observatory — Data Exploration Notebook

This notebook walks you through how `data_loader.py` works — from raw Apache log text to a clean pandas DataFrame ready for visualization.

**Goal:** By the end, you will understand the dataset well enough to build a simple Streamlit dashboard.

---
**Prerequisites**
```
pip install pandas plotly user-agents
```
You need `weblog.txt` in the same directory as this notebook (or adjust `LOG_PATH` below).

## 1  Setup — imports and file path

In [22]:
# pip install pandas plotly user-agents

In [1]:
import os
import re

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "browser"

# Adjust this path if weblog.txt lives elsewhere
LOG_PATH = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'weblog.txt')
print('Log file exists:', os.path.exists(LOG_PATH))
print('Size (MB):', round(os.path.getsize(LOG_PATH) / 1_000_000, 1))

Log file exists: True
Size (MB): 45.1


## 2  Understanding the Raw Log Format

Apache Combined Log Format looks like this:

```
IP - - [DD/Mon/YYYY:HH:MM:SS +TZ] "METHOD /path HTTP/1.x" STATUS SIZE "REFERER" "USER-AGENT"
```

We use a **regex** to pull each field out of every line.

In [2]:
# Peek at the first 3 raw lines
with open(LOG_PATH, 'r', encoding='utf-8', errors='ignore') as f:
    for i, line in enumerate(f):
        print(line.rstrip())
        if i >= 2:
            break

192.168.1.1 - - [17/Jul/2023:21:16:28 -0400] "GET / HTTP/1.1" 200 523 "-" "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/115.0"
192.168.1.1 - - [17/Jul/2023:21:16:28 -0400] "GET /css/ait.css HTTP/1.1" 200 7749 "https://huaxia.ai/" "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/115.0"
192.168.1.1 - - [17/Jul/2023:21:16:28 -0400] "GET /css/bootstrap.css HTTP/1.1" 200 145969 "https://huaxia.ai/" "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/115.0"


In [4]:
# The regex that matches every field
# Removed the incorrect initial LOG_PATTERN definition.

# Re-define with corrected grouping
LOG_PATTERN = re.compile(
    r'(?P<ip>\S+) \S+ \S+ '
    r'\[(?P<datetime>[^\]]+)\] '
    r'"(?P<method>\S+) (?P<url>\S+) \S+" '
    r'(?P<status>\d{3}) (?P<size>\S+) '
    r'"(?P<referer>[^"]*)" '
    r'"(?P<user_agent>[^"]*)"'
)

# Test it on one line
with open(LOG_PATH, 'r', encoding='utf-8', errors='ignore') as f:
    sample_line = f.readline()

m = LOG_PATTERN.match(sample_line)
if m:
    print('Parsed fields:')
    for k, v in m.groupdict().items():
        print(f'  {k:12s}: {v}')
else:
    print('No match — check that LOG_PATH points to a valid Apache log')

Parsed fields:
  ip          : 192.168.1.1
  datetime    : 17/Jul/2023:21:16:28 -0400
  method      : GET
  url         : /
  status      : 200
  size        : 523
  referer     : -
  user_agent  : Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/115.0


## 3  Parse the Entire Log into a DataFrame


In [6]:
%%time
records = []
with open(LOG_PATH, 'r', encoding='utf-8', errors='ignore') as f:
    for line in f:
        m = LOG_PATTERN.match(line)
        if m:
            d = m.groupdict()
            try:
                size = int(d['size']) if d['size'] != '-' else 0
            except ValueError:
                size = 0
            records.append({
                'ip':           d['ip'],
                'datetime_str': d['datetime'],
                'method':       d['method'],
                'url':          d['url'],
                'status':       int(d['status']),
                'size':         size,
                'referer':      d['referer'],
                'user_agent':   d['user_agent'],
            })

df_raw = pd.DataFrame(records)
print(f'Parsed {len(df_raw):,} log lines  ({len(df_raw.columns)} columns)')
df_raw.head(3)

Parsed 202,255 log lines  (8 columns)
CPU times: user 545 ms, sys: 91.2 ms, total: 636 ms
Wall time: 641 ms


,ip,datetime_str,method,url,status,size,referer,user_agent
0,192.168.1.1,17/Jul/2023:21:16:28 -0400,GET,/,200,523,-,Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:109...
1,192.168.1.1,17/Jul/2023:21:16:28 -0400,GET,/css/ait.css,200,7749,https://huaxia.ai/,Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:109...
2,192.168.1.1,17/Jul/2023:21:16:28 -0400,GET,/css/bootstrap.css,200,145969,https://huaxia.ai/,Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:109...


## 4  Feature Engineering — derived columns

The raw fields aren't very useful on their own. We add:
- **timestamp** — proper datetime so we can group by hour, day, month
- **is_static** — True for CSS/JS/images (decorative files, not content)
- **is_bot** — True for known web crawlers
- **status_cat** — groups 200, 301, 404… into readable buckets
- **section** — first URL path segment (e.g. `/course`, `/account`)

In [7]:
# ── Timestamp parsing ────────────────────────────────────────────────────────
df = df_raw.copy()
df['timestamp'] = pd.to_datetime(
    df['datetime_str'],
    format='%d/%b/%Y:%H:%M:%S %z',
    errors='coerce',
    utc=True
)
df = df.dropna(subset=['timestamp'])  # drop unparseable lines

df['date']        = df['timestamp'].dt.date
df['hour']        = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek   # 0 = Monday
df['month']       = df['timestamp'].dt.to_period('M').astype(str)

print('Date range:', df['timestamp'].min().date(), '→', df['timestamp'].max().date())
df[['timestamp', 'date', 'hour', 'day_of_week', 'month']].head(3)

Date range: 2023-07-18 → 2026-01-28


/tmp/ipykernel_24856/2343102772.py:14: UserWarning:

Converting to PeriodArray/Index representation will drop timezone information.



,timestamp,date,hour,day_of_week,month
0,2023-07-18 01:16:28+00:00,2023-07-18,1,1,2023-07
1,2023-07-18 01:16:28+00:00,2023-07-18,1,1,2023-07
2,2023-07-18 01:16:28+00:00,2023-07-18,1,1,2023-07


In [8]:
# ── Static resource detection ────────────────────────────────────────────────
STATIC_EXTENSIONS = {
    '.css', '.js', '.png', '.jpg', '.jpeg', '.gif', '.ico',
    '.svg', '.woff', '.woff2', '.ttf', '.eot', '.map',
}

def is_static(url):
    lower = url.lower()
    if os.path.splitext(lower)[1] in STATIC_EXTENSIONS:
        return True
    if lower.startswith(('/css/', '/js/', '/fonts/', '/images/', '/static/', '/socket.io/')):
        return True
    return 'favicon' in lower

df['is_static'] = df['url'].apply(is_static)

# ── Bot detection ────────────────────────────────────────────────────────────
BOT_PATTERNS = re.compile(
    r'bot|crawler|spider|crawl|slurp|bingbot|googlebot|feedcrawler|'
    r'yandex|baidu|semrush|ahref|mj12|dotbot|petalbot', re.I
)

df['is_bot'] = df['user_agent'].apply(lambda ua: bool(BOT_PATTERNS.search(ua)))

# ── Status categories ────────────────────────────────────────────────────────
df['status_cat'] = pd.cut(
    df['status'],
    bins=[0, 199, 299, 399, 499, 599],
    labels=['1xx Info', '2xx Success', '3xx Redirect', '4xx Client Err', '5xx Server Err']
)

# ── Section (first URL segment) ───────────────────────────────────────────────
df['section'] = df['url'].apply(
    lambda u: '/' + u.strip('/').split('/')[0] if u != '/' else '/'
)

print(f'Total rows: {len(df):,}')
print(f'Static requests : {df["is_static"].sum():,} ({df["is_static"].mean():.1%})')
print(f'Bot requests    : {df["is_bot"].sum():,} ({df["is_bot"].mean():.1%})')
df[['url', 'status', 'status_cat', 'is_static', 'is_bot', 'section']].head(5)

Total rows: 202,255
Static requests : 68,828 (34.0%)
Bot requests    : 12,182 (6.0%)


,url,status,status_cat,is_static,is_bot,section
0,/,200,2xx Success,False,False,/
1,/css/ait.css,200,2xx Success,True,False,/css
2,/css/bootstrap.css,200,2xx Success,True,False,/css
3,/js/bootstrap.min.js,200,2xx Success,True,False,/js
4,/js/jquery.min.js,200,2xx Success,True,False,/js


## 5  DataFrame Schema Overview

Here's the full schema your Streamlit app will receive.

In [9]:
# Column types and non-null counts at a glance
df.dtypes.to_frame('dtype').assign(non_null=df.count())

,dtype,non_null
ip,object,202255
datetime_str,object,202255
method,object,202255
url,object,202255
status,int64,202255
size,int64,202255
referer,object,202255
user_agent,object,202255
timestamp,"datetime64[ns, UTC]",202255
date,object,202255


In [10]:
# Quick numeric summary of the most useful columns
df[['status', 'size', 'hour', 'day_of_week']].describe().round(1)

,status,size,hour,day_of_week
count,202255.0,2.022550e+05,202255.0,202255.0
mean,279.8,2.819877e+05,13.2,3.0
std,89.2,2.521319e+07,7.4,2.0
min,200.0,0.000000e+00,0.0,0.0
25%,200.0,4.500000e+01,7.0,1.0
50%,200.0,1.139000e+03,15.0,3.0
75%,404.0,8.163000e+03,19.0,5.0
max,502.0,6.753718e+09,23.0,6.0


## 6  Define Useful Filtered Subsets

The main app uses three slices of the data:

| Variable | Filter | Purpose |
|----------|--------|---------|
| `df` | none | full dataset — AI tools, bot analysis |
| `df_pages` | `is_static == False` | content pages only |
| `df_humans` | `is_static == False` AND `is_bot == False` | real human visits |

In [11]:
df_pages  = df[~df['is_static']]
df_humans = df_pages[~df_pages['is_bot']]

print(f'df        (all requests)   : {len(df):>8,} rows')
print(f'df_pages  (content only)   : {len(df_pages):>8,} rows  ({len(df_pages)/len(df):.1%} of all)')
print(f'df_humans (humans + pages) : {len(df_humans):>8,} rows  ({len(df_humans)/len(df):.1%} of all)')

df        (all requests)   :  202,255 rows
df_pages  (content only)   :  133,427 rows  (66.0% of all)
df_humans (humans + pages) :  124,946 rows  (61.8% of all)


## 7  Key Metrics — KPIs your dashboard should show

In [12]:
total_requests  = len(df)
unique_visitors = df['ip'].nunique()
bandwidth_gb    = df['size'].sum() / 1e9
error_rate      = (df['status'] >= 400).mean()
bot_pct         = df['is_bot'].mean()

print(f'Total requests  : {total_requests:,}')
print(f'Unique IPs      : {unique_visitors:,}')
print(f'Bandwidth (GB)  : {bandwidth_gb:.2f}')
print(f'Error rate      : {error_rate:.1%}')
print(f'Bot traffic     : {bot_pct:.1%}')

Total requests  : 202,255
Unique IPs      : 9,134
Bandwidth (GB)  : 57.03
Error rate      : 28.6%
Bot traffic     : 6.0%


## 8  Visualizations — charts you can drop straight into Streamlit

Each block below produces a Plotly chart. In Streamlit you swap `fig.show()` for `st.plotly_chart(fig, use_container_width=True)`.

### 8a  Daily traffic trend

In [15]:
daily = df_humans.groupby('date').size().reset_index(name='requests')

fig = px.line(
    daily, x='date', y='requests',
    title='Daily Human Page Requests',
    labels={'date': 'Date', 'requests': 'Requests'},
)
fig.show()

### 8b  HTTP status code breakdown

In [14]:
status_counts = df['status_cat'].value_counts().reset_index()
status_counts.columns = ['category', 'count']

fig = px.pie(
    status_counts, names='category', values='count',
    title='HTTP Status Code Distribution',
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.show()

### 8c  Top 10 requested pages (humans only)

In [15]:
top_pages = (
    df_humans['url']
    .value_counts()
    .head(10)
    .reset_index()
)
top_pages.columns = ['url', 'hits']

fig = px.bar(
    top_pages.sort_values('hits'), x='hits', y='url',
    orientation='h',
    title='Top 10 Pages (Human Traffic)',
    labels={'hits': 'Requests', 'url': 'URL'},
)
fig.show()

### 8d  Hourly activity heatmap (day-of-week × hour)

In [16]:
heatmap_data = (
    df_humans
    .groupby(['day_of_week', 'hour'])
    .size()
    .reset_index(name='requests')
    .pivot(index='day_of_week', columns='hour', values='requests')
    .fillna(0)
)

day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

fig = px.imshow(
    heatmap_data,
    labels=dict(x='Hour of Day', y='Day of Week', color='Requests'),
    y=day_labels,
    title='Traffic Heatmap — Day vs Hour (Human Traffic)',
    color_continuous_scale='Blues',
    aspect='auto',
)
fig.show()

### 8e  Bot vs Human traffic split

In [17]:
traffic_type = (
    df_pages
    .assign(type=df_pages['is_bot'].map({True: 'Bot', False: 'Human'}))
    .groupby('type')
    .size()
    .reset_index(name='requests')
)

fig = px.pie(
    traffic_type, names='type', values='requests',
    title='Bot vs Human Traffic (Content Pages)',
    color_discrete_map={'Human': '#4ade80', 'Bot': '#f87171'},
)
fig.show()

### 8f  Monthly bandwidth trend (GB)

In [18]:
monthly_bw = (
    df
    .groupby('month')['size']
    .sum()
    .div(1e9)  # bytes → GB
    .reset_index()
    .rename(columns={'size': 'bandwidth_gb'})
)

fig = px.bar(
    monthly_bw, x='month', y='bandwidth_gb',
    title='Monthly Bandwidth (GB)',
    labels={'month': 'Month', 'bandwidth_gb': 'GB Served'},
)
fig.show()

### 8g  Top error pages

In [19]:
errors = (
    df[df['status'] >= 400]
    .groupby(['url', 'status'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(10)
)

fig = px.bar(
    errors, x='count', y='url', color='status',
    orientation='h',
    title='Top 10 Error URLs',
    labels={'count': 'Occurrences', 'url': 'URL', 'status': 'HTTP Status'},
    color_continuous_scale='Reds',
)
fig.show()

## 9  Visitor Engagement Segments

Classify human visitors by how many page requests they made:

| Segment | Requests per IP |
|---------|----------------|
| Bounced | 1 |
| Casual | 2–4 |
| Engaged | 5–15 |
| Power User | 16+ |

In [20]:
visit_counts = df_humans.groupby('ip').size()

def segment(n):
    if n == 1:   return 'Bounced'
    if n <= 4:   return 'Casual'
    if n <= 15:  return 'Engaged'
    return 'Power User'

segments = visit_counts.apply(segment).value_counts().reset_index()
segments.columns = ['segment', 'visitors']

order = ['Bounced', 'Casual', 'Engaged', 'Power User']
segments['segment'] = pd.Categorical(segments['segment'], categories=order, ordered=True)
segments = segments.sort_values('segment')

fig = px.bar(
    segments, x='segment', y='visitors',
    title='Visitor Engagement Segments',
    color='segment',
    color_discrete_sequence=px.colors.qualitative.Pastel,
)
fig.show()

segments

,segment,visitors
0,Bounced,3750
1,Casual,1575
2,Engaged,560
3,Power User,541
